# Dog Breed Identification (Local Optimized)

This notebook is optimized for local execution on Windows/Linux with GPU support.
It uses `tf.data` API for efficient data loading and Mixed Precision training.

In [ ]:
!pip install flask tensorflow numpy matplotlib pandas

import tensorflow as tf
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
from glob import glob

print(f"TensorFlow Version: {tf.__version__}")

# Check for GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Detected: {gpus}")
    try:
        # Enable Mixed Precision
        from tensorflow.keras import mixed_precision
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        print("Mixed precision enabled")
    except Exception as e:
        print(f"Error enabling mixed precision: {e}")
else:
    print("No GPU detected. Running on CPU.")

## 2. Data Preparation (Local)

Assumes data is in the `train` folder relative to this notebook.

In [ ]:
# defined paths relative to current directory
base_dir = os.getcwd()
TRAIN_PATH = os.path.join(base_dir, 'train')
LABELS_CSV = os.path.join(base_dir, 'labels.csv')

# Organize Dataset if needed
def organize_dataset(train_path, labels_path):
    if not os.path.exists(labels_path):
        print(f"Labels file not found at {labels_path}")
        return

    print(f"Checking dataset organization in {train_path}...")
    # Check for flat images
    files = [f for f in os.listdir(train_path) if f.lower().endswith('.jpg')]
    if files:
        print(f"Found {len(files)} unorganized images. Moving to folders...")
        df = pd.read_csv(labels_path)
        breeds = df['breed'].unique()
        for breed in breeds:
            os.makedirs(os.path.join(train_path, breed), exist_ok=True)
        
        params = df.set_index('id')['breed'].to_dict()
        for img in files:
            img_id = os.path.splitext(img)[0]
            if img_id in params:
                shutil.move(os.path.join(train_path, img), 
                            os.path.join(train_path, params[img_id], img))
        print("Organization complete.")
    else:
        print("Images already organized.")

if os.path.exists(TRAIN_PATH):
    organize_dataset(TRAIN_PATH, LABELS_CSV)

# Image Config
IMAGE_SIZE = [224, 224]
BATCH_SIZE = 32

# Load TF Dataset
if os.path.exists(TRAIN_PATH):
    print("Loading Datasets...")
    training_set = tf.keras.utils.image_dataset_from_directory(
        TRAIN_PATH,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=tuple(IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
    
    validation_set = tf.keras.utils.image_dataset_from_directory(
        TRAIN_PATH,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=tuple(IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
    
    class_names = training_set.class_names
    NUM_CLASSES = len(class_names)
    print(f"Classes: {NUM_CLASSES}")
    
    # Optimization
    AUTOTUNE = tf.data.AUTOTUNE
    def preprocess(image, label):
        return tf.cast(image, tf.float32) / 255.0, label
        
    training_set = training_set.map(preprocess).cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    validation_set = validation_set.map(preprocess).cache().prefetch(buffer_size=AUTOTUNE)
else:
    print("Train path not found!")

## 3. Model Training

In [ ]:
vgg = VGG19(input_shape=IMAGE_SIZE+[3], weights='imagenet', include_top=False)
for layer in vgg.layers:
    layer.trainable = False

x = Flatten()(vgg.output)
prediction = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(inputs=vgg.input, outputs=prediction)

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

checkpoint = ModelCheckpoint('dog_breed_local_model.h5', monitor='val_accuracy', save_best_only=True)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    training_set,
    validation_data=validation_set,
    epochs=10,
    callbacks=[checkpoint, early_stop]
)

# Plot
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.legend()
plt.show()